# Manual Features from Satellite Imagery

This notebook derives per-LSOA spectral features from a Landsat 8 surface-reflectance scene (2019-08-26, London extent) and uses them as predictors for deprivation. The idea: can spectral band statistics computed over each LSOA polygon capture enough land-use signal to predict IMD scores?

### Outline

1. Load satellite data (`london_landsat.nc`) — 7 Landsat 8 bands, EPSG:4326, ~36 MB
2. Load LSOA boundaries
3. Zonal statistics — per-LSOA mean for each band
4. Assemble feature table and join to IMD
5. Cross-validated model: `IMD ~ Landsat band means`

## Data

We work with two inputs: a multi-band NetCDF raster derived from Landsat 8 and the London LSOA polygon boundaries (2020 definition) used throughout the workshop.

In [ ]:
import numpy as np
import pandas
import geopandas
import xarray as xr
import regionmask

### Satellite imagery

`london_landsat.nc` contains surface reflectance for bands B1–B7 of Landsat 8, reprojected to EPSG:4326. We use the `scipy` engine because the standard `netCDF4` C extension is not available in the WASM kernel.

In [ ]:
sr = xr.open_dataset('london_landsat.nc', engine='scipy')['SR']
sr

### LSOA boundaries

We reuse the geometry column from the embeddings GeoJSON, which already carries the 2020 LSOA definitions for London.

In [ ]:
lsoa = geopandas.read_file('uk_lsoa_london_embeds_2020.geojson')[['LSOA21CD', 'geometry']].set_index('LSOA21CD')
lsoa

## Zonal statistics

We want one row per LSOA with the mean surface reflectance for each of the 7 bands — a 7-feature vector per area.

Computing this at full resolution (~30 m pixels, 1504 × 1943 grid) is memory-intensive in the browser. We use two approaches:

- **Demo (below):** coarsen the raster 8× before masking — fast enough to run live, slightly less precise
- **Pre-computed (below):** load `landsat_features.csv`, computed offline at full resolution and bundled with the workshop

### Demo: coarsened zonal statistics

Coarsening by 8× reduces the grid from 1504 × 1943 to ~188 × 243 pixels (~240 m resolution), which is still well within LSOA size (~500 m across) and runs in seconds in the browser.

In [ ]:
sr_coarse = sr.coarsen(x=8, y=8, boundary='trim').mean()

mask = regionmask.mask_geopandas(lsoa, sr_coarse.x, sr_coarse.y)

band_means = sr_coarse.groupby(mask).mean()
band_means['mask'] = lsoa.index.values
features_demo = band_means.rename({'mask': 'LSOA21CD'}).to_pandas()
features_demo

### Full-resolution features (pre-computed)

`landsat_features.csv` was computed offline at full ~30 m resolution using `regionmask` + numpy `bincount` aggregation. Use this for the modelling section.

In [ ]:
features = pandas.read_csv('landsat_features.csv', index_col='LSOA21CD')
features